In [1]:
%pip -q install requests lxml cssselect

In [10]:
SKIP_URLS = [
    'https://arcanum.fandom.com/wiki/Locations',
    'https://arcanum.fandom.com/wiki/Characters',
    'https://arcanum.fandom.com/wiki/Arcanum:_Of_Steamworks_and_Magick_Obscura_Wiki'
]

SPLIT_URLS = [
    'https://arcanum.fandom.com/wiki/The_Main_Quest',
    'https://arcanum.fandom.com/wiki/Ancient_Gods'
]

In [ ]:
import requests
import csv
import time
from lxml import html, etree
from urllib.parse import urljoin, urlparse, unquote
import random

def fetch_page_content(page_name):
    api_url = "https://arcanum.fandom.com/api.php"
    params = {
        "action": "parse",
        "page": page_name,
        "prop": "text",
        "format": "json",
        "redirects": 1
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(api_url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        if 'error' in data:
            print(f"  API Error: {data['error'].get('info', 'Unknown')}")
            return None
            
        return data['parse']['text']['*']
        
    except Exception as e:
        print(f"  Error: {e}")
        return None

def is_element(node):
    return hasattr(node, 'tag') and not isinstance(node, (etree._Comment, etree._ProcessingInstruction))

def clean_text(elem):
    if not is_element(elem):
        return ""
    
    import copy
    temp = copy.deepcopy(elem)
    
    for selector in ['.//script', './/style', './/nav', './/span[@class="mw-editsection"]']:
        for bad in temp.xpath(selector):
            bad.getparent().remove(bad)
    
    text = temp.text_content()
    
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    return ' '.join(lines)

def get_content_root(tree):
    result = tree.xpath('//div[@class="mw-parser-output"]')
    if result:
        return result[0]
    
    result = tree.xpath('//div[@id="mw-content-text"]')
    if result:
        inner = result[0].xpath('.//div[@class="mw-parser-output"]')
        if inner:
            return inner[0]
        return result[0]
    
    if tree.tag == 'div' and 'mw-parser-output' in (tree.get('class') or ''):
        return tree
    
    return tree

def get_next_element_sibling(elem):
    current = elem.getnext()
    while current is not None and not is_element(current):
        current = current.getnext()
    return current

def extract_sections(page_url, html_content):
    if not html_content or (page_url in SKIP_URLS):
        return []
    
    page_title = page_url.split('/')[-1]
    page_title = page_title.replace('_', ' ')
    page_title = unquote(page_title)
    
    if '.mes' in page_title:
        return []
    
    tree = html.fromstring(html_content)
    root = get_content_root(tree)
        
    if root is None:
        return []
    
    sections = []
    
    children = list(root)
    
    i = 0
    while i < len(children):
        elem = children[i]
        
        if not is_element(elem) or elem.tag != 'h2':
            i += 1
            continue
        
        h2_title = clean_text(elem)
        
        if not h2_title:
            i += 1
            continue
        
        # collect siblings until next h2
        siblings = []
        j = i + 1
        while j < len(children):
            sibling = children[j]
            
            if is_element(sibling) and sibling.tag == 'h2':
                break
            if is_element(sibling):
                siblings.append(sibling)
            j += 1
        
        content_parts = []
        for el in siblings:
            if el.tag not in ['table', 'nav', 'script', 'style']:
                txt = clean_text(el)
                if txt:
                    content_parts.append(txt)
        
        
        
        full_content_text = ' '.join(content_parts)
        if full_content_text and len(full_content_text) > 20:
            title = f'{page_title}: {h2_title}'
            text = full_content_text.strip().replace('  ', ' ')
            
            full_text = title + '. ' + text
            full_text = full_text.strip().replace('..', '.')
            
            sections.append({
                'LINK': page_url,
                'TITLE': title,
                'TEXT': text,
                'FULL_TEXT': full_text
            })
        
        i = j
        
    if len(sections) == 0:
        raw_texts = []
        for c in children:
            if 'toc' in (c.get('class') or ''):
                continue
            
            text = clean_text(c)
            
            if text:
                raw_texts.append(text)
            
        raw_text = '. '.join(raw_texts)
            
        raw_text = raw_text.strip().replace('  ', ' ')
        raw_text = raw_text.strip().replace('..', '.')

        
        if 'This article is a stub. You can help Arcanum: Of Steamworks and Magick Obscura Wiki by expanding it.' not in raw_text:
            sections.append({
                'LINK': page_url,
                'TITLE': page_title,
                'TEXT': raw_text,
                'FULL_TEXT': page_title + raw_text
            })
    
    return sections

def get_quest_urls():
    html_content = fetch_page_content("Quests")
    if not html_content:
        return []
    
    tree = html.fromstring(html_content)
    urls = []
    
    for elem in tree.cssselect('td:last-child > a'):
        href = elem.get('href')
        if href and '/wiki/' in href:
            clean = href.split('?')[0].split('#')[0]
            url = urljoin("https://arcanum.fandom.com", clean)
            if url not in urls and 'Quests' not in url:
                urls.append(url)
    
    for elem in tree.xpath('//li//a[@href[starts-with(.,"/wiki")]]'):
        href = elem.get('href')
        if href:
            clean = href.split('?')[0].split('#')[0]
            url = urljoin("https://arcanum.fandom.com", clean)
            if url not in urls and 'Quests' not in url:
                urls.append(url)
    
    print(f"Found {len(urls)} unique quest URLs")
    return urls

def process_quests(quest_urls, output_file='arcanum_quests.csv'):
    all_data = []
    
    for idx, url in enumerate(quest_urls, 1):
        print(f"\n[{idx}/{len(quest_urls)}] {url}")
        
        parsed = urlparse(url)
        page_name = unquote(parsed.path.split('/')[-1])
        if not page_name:
            continue
        
        html_content = fetch_page_content(page_name)
        if not html_content:
            continue
        
        sections = extract_sections(url, html_content)
        if sections:
            print(f"  ✓ {len(sections)} sections")
            all_data.extend(sections)
        else:
            print(f"  ✗ No sections")
        
        time.sleep(random.uniform(0.3, 0.8))
    
    if all_data:
        with open(output_file, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=['LINK', 'TITLE', 'TEXT', 'FULL_TEXT'])
            writer.writeheader()
            writer.writerows(all_data)
        print(f"\n✓ Saved {len(all_data)} sections to {output_file}")
    else:
        print("\n✗ No data extracted")
        
    return all_data

In [39]:
# urls = get_quest_urls()

result = process_quests(['https://arcanum.fandom.com/wiki/Ancient_Gods'], 'temp.csv')

full = '\n'.join([item['TEXT'] for item in result])

print(full)

print('\n', len(full))


[1/1] https://arcanum.fandom.com/wiki/Ancient_Gods
  ✓ 13 sections

✓ Saved 13 sections to temp.csv
The pantheon of the Ancient Gods consists of 8 Lesser Gods (one for each of the sentient Races); 3 Greater Gods (one for each point of the balance of Alignment); and the All Father, ruling over the 11 others. A few of the ancient gods still have some followers, but most of the ancient gods have faded into obscurity. These have been replaced to some extent by cults of personality to historic figures like Nasrudin, Arronax, and Kerghan,
Each god has an altar, and each altar can accept a particular offering, and each offering grants a particular blessing. Offerings to the lesser gods can be made at any time. Offerings to the greater gods and the All Father require special conditions.
All 12 gods are interconnected in a way not fully understood by contemporary scholars. The web of relationships is called "Mazzerin's Mystery", after an historic Elven mystic who attempted to unravel the threa

In [2]:
import pandas as pd

pd.read_csv('./arcanum_quests.csv')

,LINK,TITLE,TEXT
0,https://arcanum.fandom.com/wiki/Find_the_Bow_o...,Find the Bow of Ecclesiastes: Walkthrough,Black Root Kietzel Pierce can be found in Blac...
1,https://arcanum.fandom.com/wiki/Find_the_Bow_o...,Find the Bow of Ecclesiastes: Notes,This quest can be done in tandem with the Find...
2,https://arcanum.fandom.com/wiki/Kill_Sir_Garri...,"Kill Sir Garrick Stout: Path 1, Dodge Mastery ...",This path is for players who only desire Dodge...
3,https://arcanum.fandom.com/wiki/Find_Lady_Druella,"Find Lady Druella: Path 2, Melee Mastery only",This path is for players who only desire Melee...
4,https://arcanum.fandom.com/wiki/Retrieve_Azram...,Retrieve Azram's Star: Walkthrough,Black Root Clarissa Shalmo can be found in Bla...
...,...,...,...
224,https://arcanum.fandom.com/wiki/Thrayne_Wants_...,Thrayne Wants His Brother to Return Home: Shor...,"After talking with Thrayne Iron Heart, he will..."
225,https://arcanum.fandom.com/wiki/Defeat_Kerghan,Defeat Kerghan: Short Description,143 Arronax: Defeat Kerghan. (Prereq: Q142. Re...
226,https://arcanum.fandom.com/wiki/Free_Arronax,Free Arronax: Short Description,142 Arronax: Effect Arronax’s release from mag...
227,https://arcanum.fandom.com/wiki/Kill_the_Banished,Kill the Banished: Short Description,"144 Kerghan: Kill Arronax, The Bane of Kree, G..."
